# 🧬 16S rRNA Amplicon Sequencing Data Analysis
## Student-Driven Microbiome Exploration — FSU Undergraduate Curriculum
**Department of Chemistry and Biochemistry, Florida State University**

---

This notebook walks you through the complete **bioinformatic analysis pipeline** for 16S rRNA amplicon sequencing data generated from the FSU low-cost microbiome curriculum.  
By the end, you will have:
- ✅ Imported and quality-checked your raw FASTQ files
- ✅ Denoised reads into ASVs using DADA2
- ✅ Assigned taxonomy with the SILVA 138 classifier
- ✅ Calculated α- and β-diversity metrics
- ✅ Generated publication-quality figures (bar plots, PCoA, rarefaction curves)
- ✅ Run differential abundance analysis (ANCOM-BC)

> **⏱ Estimated total runtime:** ~2–3 hours for a typical class dataset (20–40 samples)  
> **💾 Runtime required:** CPU (no GPU needed)  
> **📦 QIIME 2 version:** `qiime2-amplicon-2024.5`

---
### 📋 How to use this notebook
1. Run cells **in order** from top to bottom.
2. Cells marked with `# @title` have a ▶ button — click to expand and run.
3. Edit only the **USER CONFIGURATION** sections (clearly marked with 🔧).
4. Save your work frequently to Google Drive.
5. If the runtime disconnects, re-run from **Section 1** (environment setup).


---
## Section 0 — Before You Begin 📌

### What you need before running this notebook

| Item | Description |
|------|-------------|
| **Raw FASTQ files** | Paired-end (R1 + R2) `.fastq.gz` files from Illumina MiSeq, one pair per sample |
| **Manifest file** | A `.tsv` file mapping sample IDs → file paths (see format below) |
| **Metadata file** | A `.tsv` file with your experimental variables (see format below) |
| **Google Drive** | Files must be uploaded to your Drive before starting |

---
### Manifest file format (`manifest.tsv`)
```
sample-id	forward-absolute-filepath	reverse-absolute-filepath
sample-1	/content/drive/MyDrive/my_project/sample1_R1.fastq.gz	/content/drive/MyDrive/my_project/sample1_R2.fastq.gz
sample-2	/content/drive/MyDrive/my_project/sample2_R1.fastq.gz	/content/drive/MyDrive/my_project/sample2_R2.fastq.gz
```
> 💡 **Tip:** Create this in Excel → Save As → Tab Delimited Text (.txt), then rename to `.tsv`

---
### Metadata file format (`metadata.tsv`)
```
sample-id	group	location	time_point
sample-1	treatment	fountain	before
sample-2	control	fountain	after
```
> ⚠️ **Rules:** Use only letters, numbers, and underscores. No spaces or special characters.  
> QIIME 2 is **case-sensitive** — `Before` ≠ `before`.

---
### How to upload files to Google Drive
1. Go to [drive.google.com](https://drive.google.com)
2. Create a folder (e.g., `my_microbiome_project`)
3. Upload your FASTQ files, manifest.tsv, and metadata.tsv into that folder
4. Note the full path: `/content/drive/MyDrive/my_microbiome_project/`


---
## Section 1 — Environment Setup ⚙️
> **Run this every time you start a new Colab session.**  
> Installation takes approximately **15–25 minutes**. Get a coffee! ☕


In [ ]:
# @title Step 1.1 — Mount Google Drive { display-mode: "form" }
from google.colab import drive
drive.mount('/content/drive')
print("✅ Google Drive mounted successfully!")
print("   Your files are accessible at: /content/drive/MyDrive/")


In [ ]:
# @title Step 1.2 — Install Miniforge (conda) { display-mode: "form" }
# This installs a lightweight conda for Colab — takes ~2 min
import os, subprocess

print("Installing Miniforge...")
result = subprocess.run(
    "wget -q https://github.com/conda-forge/miniforge/releases/latest/download/Miniforge3-Linux-x86_64.sh -O /tmp/miniforge.sh && bash /tmp/miniforge.sh -b -p /opt/miniforge",
    shell=True, capture_output=True, text=True
)
if result.returncode == 0:
    print("✅ Miniforge installed successfully!")
else:
    print("❌ Error:", result.stderr[:500])

# Add conda to PATH
os.environ['PATH'] = '/opt/miniforge/bin:' + os.environ['PATH']
print("   conda version:", subprocess.run("conda --version", shell=True, capture_output=True, text=True).stdout.strip())


In [ ]:
# @title Step 1.3 — Install QIIME 2 { display-mode: "form" }
# This is the longest step — approximately 15-20 minutes
import subprocess, os

os.environ['PATH'] = '/opt/miniforge/bin:' + os.environ['PATH']

print("Installing QIIME 2 amplicon distribution...")
print("(This will take 15-20 minutes — please be patient)")
print("-" * 50)

# Create the QIIME 2 environment
install_cmd = '''
conda create -y -n qiime2-amplicon-2024.5 -c conda-forge -c bioconda -c qiime2 -c defaults \
  qiime2=2024.5 q2-dada2 q2-diversity q2-taxa q2-feature-classifier \
  q2-cutadapt q2-composition q2-phylogeny q2-alignment 2>&1
'''

result = subprocess.run(install_cmd, shell=True, executable='/bin/bash',
                        capture_output=True, text=True, timeout=2400)

if "Successfully installed" in result.stdout or result.returncode == 0:
    print("✅ QIIME 2 installed successfully!")
else:
    # Try alternative lightweight install method
    print("Trying alternative install method...")
    result2 = subprocess.run(
        "pip install -q qiime2 2>&1 | tail -3",
        shell=True, capture_output=True, text=True
    )
    print(result2.stdout)

# Verify installation
verify = subprocess.run(
    "source activate qiime2-amplicon-2024.5 && qiime --version",
    shell=True, executable='/bin/bash', capture_output=True, text=True
)
if verify.returncode == 0:
    print("✅ QIIME 2 version:", verify.stdout.strip())
else:
    print("⚠️  Run Section 1.4 for alternative setup")


In [ ]:
# @title Step 1.4 — Install Python analysis packages { display-mode: "form" }
# Quick install of visualization and statistics packages (~2 min)
!pip install -q pandas numpy matplotlib seaborn scipy scikit-bio biom-format qiime2 2>/dev/null

# Verify key imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# Set plotting style for publication-quality figures
plt.rcParams.update({
    'figure.dpi': 150,
    'font.family': 'sans-serif',
    'font.size': 11,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': False
})

print("✅ All Python packages imported successfully!")
print(f"   pandas {pd.__version__} | numpy {np.__version__} | matplotlib {matplotlib.__version__}")


In [ ]:
# @title Step 1.5 — Helper function to run QIIME 2 commands { display-mode: "form" }
import subprocess, sys, os

QIIME_ENV = "qiime2-amplicon-2024.5"

def run_qiime(cmd, description=""):
    """Run a QIIME 2 command and display progress."""
    full_cmd = f"source /opt/miniforge/etc/profile.d/conda.sh && conda activate {QIIME_ENV} && {cmd}"
    
    print(f"▶ {description or 'Running QIIME 2 command...'}")
    print(f"  Command: {cmd[:80]}{'...' if len(cmd) > 80 else ''}")
    
    result = subprocess.run(full_cmd, shell=True, executable='/bin/bash',
                            capture_output=True, text=True)
    
    if result.returncode == 0:
        print(f"✅ Done!")
        if result.stdout.strip():
            print("   Output:", result.stdout.strip()[:200])
    else:
        print(f"❌ Error (return code {result.returncode}):")
        print("   STDERR:", result.stderr[-500:] if result.stderr else "none")
        print("   STDOUT:", result.stdout[-300:] if result.stdout else "none")
    
    return result

print("✅ Helper functions ready!")


---
## Section 2 — 🔧 User Configuration
> **Edit this section to match your project.** These variables are used throughout the notebook.


In [ ]:
# @title Step 2.1 — Set your project parameters { display-mode: "form" }
import os

# ============================================================
# 🔧 EDIT THESE VALUES FOR YOUR PROJECT
# ============================================================

PROJECT_NAME = "my_microbiome_project"       # No spaces — use underscores

# --- File Paths (adjust to your Google Drive structure) ---
BASE_DIR     = f"/content/drive/MyDrive/{PROJECT_NAME}"
MANIFEST     = f"{BASE_DIR}/manifest.tsv"
METADATA     = f"{BASE_DIR}/metadata.tsv"

# --- SILVA Classifier path (download in Section 2.2) ---
CLASSIFIER   = f"/content/drive/MyDrive/classifiers/silva-138-99-nb-classifier.qza"

# --- Sequencing Parameters ---
READ_TYPE    = "paired-end"   # "paired-end" or "single-end"
PRIMER_F     = "GTGYCAGCMGCCGCGGTAA"    # 515F (V4)
PRIMER_R     = "GGACTACNVGGGTWTCTAAT"   # 806R (V4)
TRUNC_LEN_F  = 250    # Where to truncate forward reads (check quality plots!)
TRUNC_LEN_R  = 200    # Where to truncate reverse reads

# --- Diversity Analysis ---
SAMPLING_DEPTH = 10000   # Rarefaction depth — use minimum sample depth
                         # (you'll refine this after Step 4)

# --- Differential Abundance (ANCOM-BC) ---
FORMULA      = "group"  # Metadata column(s) for ANCOM-BC formula
                        # Use + to combine: "group+time_point"

# --- Samples to exclude (if any found to be low quality) ---
EXCLUDE_IDS  = []       # e.g., ["sample-5", "sample-12"]

# ============================================================
# DO NOT EDIT BELOW THIS LINE
# ============================================================
WORK_DIR = f"/content/qiime2_work/{PROJECT_NAME}"
os.makedirs(WORK_DIR, exist_ok=True)

# Create project subdirectories
for d in ["02_import","03_cutadapt","03_quality","04_dada2","05_taxonomy",
          "06_phylogeny","07_diversity","08_ancombc","09_figures"]:
    os.makedirs(f"{WORK_DIR}/{d}", exist_ok=True)

print("=" * 55)
print(f"  Project:         {PROJECT_NAME}")
print(f"  Base directory:  {BASE_DIR}")
print(f"  Manifest:        {MANIFEST}")
print(f"  Metadata:        {METADATA}")
print(f"  Read type:       {READ_TYPE}")
print(f"  Trunc F / R:     {TRUNC_LEN_F} / {TRUNC_LEN_R}")
print(f"  Sampling depth:  {SAMPLING_DEPTH}")
print(f"  ANCOM-BC formula: {FORMULA}")
print("=" * 55)

# Verify that key files exist
for label, path in [("Manifest", MANIFEST), ("Metadata", METADATA)]:
    if os.path.exists(path):
        print(f"✅ {label} found: {path}")
    else:
        print(f"⚠️  {label} NOT FOUND: {path}  ← Fix this path!")


In [ ]:
# @title Step 2.2 — Download SILVA 138 classifier (if needed) { display-mode: "form" }
import os, subprocess

classifier_dir = "/content/drive/MyDrive/classifiers"
classifier_path = f"{classifier_dir}/silva-138-99-nb-classifier.qza"
classifier_url = "https://data.qiime2.org/2024.5/common/silva-138-99-nb-classifier.qza"

os.makedirs(classifier_dir, exist_ok=True)

if os.path.exists(classifier_path):
    size_mb = os.path.getsize(classifier_path) / 1e6
    print(f"✅ Classifier already exists ({size_mb:.0f} MB) — skipping download.")
else:
    print("Downloading SILVA 138 classifier (~1 GB)...")
    print("⏱ This takes 5–15 minutes depending on connection speed.")
    result = subprocess.run(
        f"wget -q --show-progress -O {classifier_path} {classifier_url}",
        shell=True, capture_output=False
    )
    if result.returncode == 0:
        size_mb = os.path.getsize(classifier_path) / 1e6
        print(f"\n✅ Downloaded successfully ({size_mb:.0f} MB)")
        print(f"   Saved to Drive — won't need to download again!")
    else:
        print("❌ Download failed. Try manually downloading from:")
        print(f"   {classifier_url}")
        print(f"   and uploading to {classifier_dir}/")


In [ ]:
# @title Step 2.3 — Preview your metadata file { display-mode: "form" }
import pandas as pd

try:
    meta = pd.read_csv(METADATA, sep='\t')
    print(f"✅ Metadata loaded: {meta.shape[0]} samples × {meta.shape[1]} columns")
    print()
    print("Preview:")
    display(meta.head(10))
    print()
    print("Column names:", list(meta.columns))
    print()
    print("Sample counts per group:")
    for col in meta.columns[1:]:
        if meta[col].nunique() <= 10:
            print(f"  [{col}]:", dict(meta[col].value_counts()))
except FileNotFoundError:
    print(f"❌ Metadata file not found: {METADATA}")
    print("   Please upload your metadata.tsv to Google Drive first.")


---
## Section 3 — Data Import & Quality Control 🔬

**What happens here:**
1. Import your raw FASTQ files into QIIME 2 format (`.qza` artifact)
2. Trim primer sequences with `cutadapt`
3. Visualize read quality to determine truncation parameters


In [ ]:
# @title Step 3.1 — Import FASTQ files into QIIME 2 { display-mode: "form" }

input_format = ("PairedEndFastqManifestPhred33V2" if READ_TYPE == "paired-end"
                else "SingleEndFastqManifestPhred33V2")
seq_type     = ("SampleData[PairedEndSequencesWithQuality]" if READ_TYPE == "paired-end"
                else "SampleData[SequencesWithQuality]")

import_cmd = f"""qiime tools import \
  --type '{seq_type}' \
  --input-path {MANIFEST} \
  --output-path {WORK_DIR}/02_import/demux.qza \
  --input-format {input_format}"""

run_qiime(import_cmd, "Importing FASTQ files into QIIME 2 format")


In [ ]:
# @title Step 3.2 — Summarize imported data { display-mode: "form" }
summarize_cmd = f"""qiime demux summarize \
  --i-data {WORK_DIR}/02_import/demux.qza \
  --o-visualization {WORK_DIR}/03_quality/demux_summary.qzv"""

run_qiime(summarize_cmd, "Generating quality summary")

print()
print("📊 To view quality plots:")
print("   1. Download the file:", f"{WORK_DIR}/03_quality/demux_summary.qzv")
print("   2. Open: https://view.qiime2.org")
print("   3. Drag and drop the .qzv file")
print()
print("🔧 Use the quality plots to set TRUNC_LEN_F and TRUNC_LEN_R in Section 2!")
print("   → Look for where quality scores drop below ~Q25")


In [ ]:
# @title Step 3.3 — Trim primers with cutadapt { display-mode: "form" }
if READ_TYPE == "paired-end":
    trim_cmd = f"""qiime cutadapt trim-paired \
  --i-demultiplexed-sequences {WORK_DIR}/02_import/demux.qza \
  --p-front-f {PRIMER_F} \
  --p-front-r {PRIMER_R} \
  --p-discard-untrimmed \
  --p-minimum-length 100 \
  --o-trimmed-sequences {WORK_DIR}/03_cutadapt/demux_trimmed.qza \
  --verbose 2>{WORK_DIR}/03_cutadapt/cutadapt_log.txt"""
else:
    trim_cmd = f"""qiime cutadapt trim-single \
  --i-demultiplexed-sequences {WORK_DIR}/02_import/demux.qza \
  --p-front {PRIMER_F} \
  --p-discard-untrimmed \
  --p-minimum-length 100 \
  --o-trimmed-sequences {WORK_DIR}/03_cutadapt/demux_trimmed.qza \
  --verbose 2>{WORK_DIR}/03_cutadapt/cutadapt_log.txt"""

run_qiime(trim_cmd, "Trimming primer sequences with cutadapt")

# Show trimming stats
import subprocess
log_result = subprocess.run(f"grep -E 'Reads written|Total reads' {WORK_DIR}/03_cutadapt/cutadapt_log.txt | head -20",
                            shell=True, capture_output=True, text=True)
if log_result.stdout:
    print("\nTrimming summary (first 5 samples):")
    print(log_result.stdout[:800])


---
## Section 4 — DADA2 Denoising 🧹

**What is DADA2?**  
DADA2 is a statistical algorithm that corrects sequencing errors and generates **Amplicon Sequence Variants (ASVs)** — exact sequences rather than clustered OTUs. This gives higher resolution for species-level identification.

**What happens here:**
1. Error modeling from your data
2. Denoising, chimera removal
3. Merging paired reads
4. Output: feature table (ASV counts per sample) + representative sequences

> ⏱ **This is the longest step — typically 30–90 minutes** depending on sample number and read depth.


In [ ]:
# @title Step 4.1 — Run DADA2 denoising { display-mode: "form" }
import multiprocessing
n_threads = multiprocessing.cpu_count()
print(f"Using {n_threads} CPU threads")

if READ_TYPE == "paired-end":
    dada2_cmd = f"""qiime dada2 denoise-paired \
  --i-demultiplexed-seqs {WORK_DIR}/03_cutadapt/demux_trimmed.qza \
  --p-trim-left-f 0 \
  --p-trim-left-r 0 \
  --p-trunc-len-f {TRUNC_LEN_F} \
  --p-trunc-len-r {TRUNC_LEN_R} \
  --p-n-threads {n_threads} \
  --o-table {WORK_DIR}/04_dada2/feature_table.qza \
  --o-representative-sequences {WORK_DIR}/04_dada2/rep_seqs.qza \
  --o-denoising-stats {WORK_DIR}/04_dada2/denoising_stats.qza \
  --verbose 2>&1 | tee {WORK_DIR}/04_dada2/dada2_log.txt"""
else:
    dada2_cmd = f"""qiime dada2 denoise-single \
  --i-demultiplexed-seqs {WORK_DIR}/03_cutadapt/demux_trimmed.qza \
  --p-trim-left 0 \
  --p-trunc-len {TRUNC_LEN_F} \
  --p-n-threads {n_threads} \
  --o-table {WORK_DIR}/04_dada2/feature_table.qza \
  --o-representative-sequences {WORK_DIR}/04_dada2/rep_seqs.qza \
  --o-denoising-stats {WORK_DIR}/04_dada2/denoising_stats.qza \
  --verbose 2>&1 | tee {WORK_DIR}/04_dada2/dada2_log.txt"""

print("Starting DADA2... (30–90 min for typical datasets)")
run_qiime(dada2_cmd, "DADA2 denoising — please be patient!")


In [ ]:
# @title Step 4.2 — Summarize DADA2 results { display-mode: "form" }
# Feature table summary
ft_cmd = f"""qiime feature-table summarize \
  --i-table {WORK_DIR}/04_dada2/feature_table.qza \
  --m-sample-metadata-file {METADATA} \
  --o-visualization {WORK_DIR}/04_dada2/feature_table_summary.qzv"""

run_qiime(ft_cmd, "Generating feature table summary")

# Denoising stats
stats_cmd = f"""qiime metadata tabulate \
  --m-input-file {WORK_DIR}/04_dada2/denoising_stats.qza \
  --o-visualization {WORK_DIR}/04_dada2/denoising_stats.qzv"""

run_qiime(stats_cmd, "Tabulating denoising statistics")

print()
print("📊 Open these at https://view.qiime2.org:")
print(f"   {WORK_DIR}/04_dada2/feature_table_summary.qzv")
print(f"     → Use this to choose your SAMPLING_DEPTH (Section 2)")
print(f"     → Look at 'Interactive Sample Detail' tab")
print(f"     → Pick a depth that retains most samples (exclude outliers)")
print(f"   {WORK_DIR}/04_dada2/denoising_stats.qzv")
print(f"     → Check % reads passing each filter stage")


In [ ]:
# @title Step 4.3 — Extract and plot denoising statistics { display-mode: "form" }
import subprocess, pandas as pd, matplotlib.pyplot as plt
import matplotlib.ticker as mticker

# Export denoising stats to TSV
export_cmd = f"""qiime tools export \
  --input-path {WORK_DIR}/04_dada2/denoising_stats.qza \
  --output-path {WORK_DIR}/04_dada2/denoising_stats_export"""

result = run_qiime(export_cmd, "Exporting denoising stats")

import os
stats_tsv = f"{WORK_DIR}/04_dada2/denoising_stats_export/stats.tsv"

if os.path.exists(stats_tsv):
    df = pd.read_csv(stats_tsv, sep='\t', skiprows=[1])
    df = df.rename(columns={'sample-id': 'sample_id'})
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Read retention bar chart
    if 'input' in df.columns and 'non-chimeric' in df.columns:
        df_sorted = df.sort_values('input', ascending=False).head(30)
        x = range(len(df_sorted))
        axes[0].bar(x, df_sorted['input'], color='#b0c4de', label='Input reads')
        axes[0].bar(x, df_sorted['non-chimeric'], color='#2e75b6', label='Non-chimeric reads')
        axes[0].set_xlabel('Sample')
        axes[0].set_ylabel('Read count')
        axes[0].set_title('Read retention per sample')
        axes[0].legend()
        axes[0].set_xticks([])
        axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
    
    # % retention histogram
    if 'input' in df.columns and 'non-chimeric' in df.columns:
        df['pct_retained'] = df['non-chimeric'] / df['input'] * 100
        axes[1].hist(df['pct_retained'].dropna(), bins=15, color='#2e75b6', edgecolor='white')
        axes[1].axvline(df['pct_retained'].median(), color='red', linestyle='--',
                       label=f"Median: {df['pct_retained'].median():.1f}%")
        axes[1].set_xlabel('% reads retained after denoising')
        axes[1].set_ylabel('Number of samples')
        axes[1].set_title('Read retention distribution')
        axes[1].legend()
    
    plt.suptitle('DADA2 Denoising Statistics', fontsize=13, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.savefig(f"{WORK_DIR}/09_figures/denoising_stats.pdf", bbox_inches='tight')
    plt.show()
    
    print(f"\n📊 Summary:")
    print(f"   Total samples: {len(df)}")
    if 'input' in df.columns:
        print(f"   Median input reads: {df['input'].median():,.0f}")
        print(f"   Median non-chimeric reads: {df['non-chimeric'].median():,.0f}")
        print(f"   Median retention: {df['pct_retained'].median():.1f}%")
        low_retention = df[df['pct_retained'] < 50]['sample_id'].tolist()
        if low_retention:
            print(f"\n⚠️  Samples with <50% retention (consider excluding):")
            for s in low_retention:
                print(f"   {s}")
else:
    print("⚠️  Stats file not found yet — run Step 4.1 first")


---
## Section 5 — Taxonomic Classification 🔍

**What happens here:**
1. Classify ASVs against the SILVA 138 database using a naive Bayes classifier
2. Filter out non-bacterial sequences (mitochondria, chloroplasts, Eukaryota)
3. Remove any low-quality samples
4. Generate taxonomic bar plots


In [ ]:
# @title Step 5.1 — Classify ASVs with SILVA 138 { display-mode: "form" }
# This step takes 20–40 minutes on Colab
import multiprocessing
n_threads = multiprocessing.cpu_count()

classify_cmd = f"""qiime feature-classifier classify-sklearn \
  --i-classifier {CLASSIFIER} \
  --i-reads {WORK_DIR}/04_dada2/rep_seqs.qza \
  --p-n-jobs {n_threads} \
  --o-classification {WORK_DIR}/05_taxonomy/taxonomy.qza"""

print("Starting taxonomic classification... (~20–40 min)")
run_qiime(classify_cmd, "SILVA 138 taxonomic classification")


In [ ]:
# @title Step 5.2 — Filter non-bacterial sequences { display-mode: "form" }
filter_taxa_cmd = f"""qiime taxa filter-table \
  --i-table {WORK_DIR}/04_dada2/feature_table.qza \
  --i-taxonomy {WORK_DIR}/05_taxonomy/taxonomy.qza \
  --p-exclude mitochondria,chloroplast,Eukaryota,Archaea,Unassigned \
  --p-include p__ \
  --o-filtered-table {WORK_DIR}/05_taxonomy/feature_table_filtered.qza"""

run_qiime(filter_taxa_cmd, "Filtering non-bacterial sequences")
print("\n   ✅ Removed: mitochondria, chloroplasts, Eukaryota, Archaea, Unassigned")


In [ ]:
# @title Step 5.3 — Remove low-quality samples (if any) { display-mode: "form" }
import os, shutil

if EXCLUDE_IDS:
    exclude_str = "','".join(EXCLUDE_IDS)
    exclude_cmd = f"""qiime feature-table filter-samples \
  --i-table {WORK_DIR}/05_taxonomy/feature_table_filtered.qza \
  --m-metadata-file {METADATA} \
  --p-exclude-ids \
  --p-where "[sample-id] IN ('{exclude_str}')" \
  --o-filtered-table {WORK_DIR}/05_taxonomy/feature_table_final.qza"""
    
    run_qiime(exclude_cmd, f"Removing {len(EXCLUDE_IDS)} excluded samples")
    
    # Create filtered metadata
    import pandas as pd
    meta = pd.read_csv(METADATA, sep='\t')
    meta_filtered = meta[~meta['sample-id'].isin(EXCLUDE_IDS)]
    meta_filtered.to_csv(f"{WORK_DIR}/05_taxonomy/metadata_filtered.tsv",
                         sep='\t', index=False)
    METADATA_FINAL = f"{WORK_DIR}/05_taxonomy/metadata_filtered.tsv"
    print(f"   Removed: {EXCLUDE_IDS}")
    print(f"   Remaining samples: {len(meta_filtered)}")
else:
    # Just copy as final
    shutil.copy(f"{WORK_DIR}/05_taxonomy/feature_table_filtered.qza",
                f"{WORK_DIR}/05_taxonomy/feature_table_final.qza")
    METADATA_FINAL = METADATA
    print("✅ No samples excluded — using all samples.")

print(f"   Final metadata: {METADATA_FINAL}")


In [ ]:
# @title Step 5.4 — Generate taxonomic bar plots { display-mode: "form" }
barplot_cmd = f"""qiime taxa barplot \
  --i-table {WORK_DIR}/05_taxonomy/feature_table_final.qza \
  --i-taxonomy {WORK_DIR}/05_taxonomy/taxonomy.qza \
  --m-metadata-file {METADATA_FINAL} \
  --o-visualization {WORK_DIR}/05_taxonomy/taxa_barplot.qzv"""

run_qiime(barplot_cmd, "Generating taxonomic bar plots")

print()
print("📊 Open the bar plots at https://view.qiime2.org:")
print(f"   {WORK_DIR}/05_taxonomy/taxa_barplot.qzv")
print("   → Use Level 2 (Phylum) or Level 6 (Genus) for your poster")
print("   → Sort by metadata column to compare groups")


In [ ]:
# @title Step 5.5 — Export taxonomy and plot top genera { display-mode: "form" }
import subprocess, os, pandas as pd, matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# Export feature table and taxonomy
export_cmds = [
    (f"""qiime tools export --input-path {WORK_DIR}/05_taxonomy/feature_table_final.qza --output-path {WORK_DIR}/05_taxonomy/export_table""",
     "Exporting feature table"),
    (f"""qiime tools export --input-path {WORK_DIR}/05_taxonomy/taxonomy.qza --output-path {WORK_DIR}/05_taxonomy/export_taxonomy""",
     "Exporting taxonomy"),
]
for cmd, desc in export_cmds:
    run_qiime(cmd, desc)

# Convert biom to tsv
biom_cmd = f"biom convert -i {WORK_DIR}/05_taxonomy/export_table/feature-table.biom -o {WORK_DIR}/05_taxonomy/feature_table.tsv --to-tsv"
subprocess.run(biom_cmd, shell=True)

table_tsv = f"{WORK_DIR}/05_taxonomy/feature_table.tsv"
tax_tsv   = f"{WORK_DIR}/05_taxonomy/export_taxonomy/taxonomy.tsv"

if os.path.exists(table_tsv) and os.path.exists(tax_tsv):
    # Load data
    ft = pd.read_csv(table_tsv, sep='\t', skiprows=1, index_col=0)
    tx = pd.read_csv(tax_tsv, sep='\t', index_col=0)
    
    # Parse genus level
    def get_genus(tax_str):
        try:
            parts = tax_str.split(';')
            for p in parts:
                if 'g__' in p and len(p.strip()) > 3:
                    return p.strip().replace('g__', '').strip()
        except:
            pass
        return 'Unknown'
    
    tx['Genus'] = tx['Taxon'].apply(get_genus)
    
    # Merge and collapse to genus
    ft_with_tax = ft.merge(tx[['Genus']], left_index=True, right_index=True, how='left')
    genus_table = ft_with_tax.groupby('Genus')[ft.columns].sum()
    genus_rel = genus_table.div(genus_table.sum(axis=0), axis=1) * 100
    
    # Top 15 genera
    top_genera = genus_rel.mean(axis=1).sort_values(ascending=False).head(15).index
    top_df = genus_rel.loc[top_genera]
    
    # Plot stacked bar
    meta = pd.read_csv(METADATA_FINAL, sep='\t', index_col='sample-id')
    samples = [s for s in top_df.columns if s in meta.index]
    
    colors = plt.cm.tab20(range(15))
    fig, ax = plt.subplots(figsize=(max(10, len(samples) * 0.5), 6))
    
    bottom = [0] * len(samples)
    for i, genus in enumerate(top_genera):
        vals = [top_df.loc[genus, s] if s in top_df.columns else 0 for s in samples]
        ax.bar(samples, vals, bottom=bottom, color=colors[i], label=genus, width=0.8)
        bottom = [b + v for b, v in zip(bottom, vals)]
    
    ax.set_xlabel('Sample')
    ax.set_ylabel('Relative abundance (%)')
    ax.set_title('Top 15 Genera — Relative Abundance', fontweight='bold')
    ax.set_xticklabels(samples, rotation=45, ha='right', fontsize=8)
    ax.legend(bbox_to_anchor=(1.01, 1), loc='upper left', fontsize=8, frameon=False)
    plt.tight_layout()
    plt.savefig(f"{WORK_DIR}/09_figures/genus_barplot.pdf", bbox_inches='tight')
    plt.show()
    print(f"✅ Saved to: {WORK_DIR}/09_figures/genus_barplot.pdf")
else:
    print("⚠️ Export files not found — run QIIME 2 export steps first")


---
## Section 6 — Phylogeny & Diversity Analysis 🌳

**Core diversity metrics computed:**

| Metric | Type | What it measures |
|--------|------|-----------------|
| Shannon index | α-diversity | Species evenness and richness within a sample |
| Observed features | α-diversity | Number of unique ASVs per sample |
| Faith's PD | α-diversity | Phylogenetic diversity within a sample |
| Bray-Curtis | β-diversity | Compositional similarity between samples |
| UniFrac (unweighted) | β-diversity | Phylogenetic similarity (presence/absence) |
| UniFrac (weighted) | β-diversity | Phylogenetic similarity (abundance-weighted) |


In [ ]:
# @title Step 6.1 — Build phylogenetic tree { display-mode: "form" }
tree_cmd = f"""qiime phylogeny align-to-tree-mafft-fasttree \
  --i-sequences {WORK_DIR}/04_dada2/rep_seqs.qza \
  --o-alignment {WORK_DIR}/06_phylogeny/aligned.qza \
  --o-masked-alignment {WORK_DIR}/06_phylogeny/masked.qza \
  --o-tree {WORK_DIR}/06_phylogeny/unrooted_tree.qza \
  --o-rooted-tree {WORK_DIR}/06_phylogeny/rooted_tree.qza \
  --p-n-threads auto"""

run_qiime(tree_cmd, "Building phylogenetic tree (MAFFT + FastTree)")


In [ ]:
# @title Step 6.2 — Compute core diversity metrics { display-mode: "form" }
print(f"Using sampling depth: {SAMPLING_DEPTH:,} reads")
print("⚠️  Samples with fewer reads than this will be EXCLUDED from diversity analysis.")
print("   Adjust SAMPLING_DEPTH in Section 2 if needed.\n")

diversity_cmd = f"""qiime diversity core-metrics-phylogenetic \
  --i-phylogeny {WORK_DIR}/06_phylogeny/rooted_tree.qza \
  --i-table {WORK_DIR}/05_taxonomy/feature_table_final.qza \
  --p-sampling-depth {SAMPLING_DEPTH} \
  --m-metadata-file {METADATA_FINAL} \
  --output-dir {WORK_DIR}/07_diversity/core_metrics"""

run_qiime(diversity_cmd, "Computing core diversity metrics (rarefying to sampling depth)")


In [ ]:
# @title Step 6.3 — Alpha diversity statistical tests { display-mode: "form" }
alpha_metrics = ['shannon_vector', 'observed_features_vector', 'faith_pd_vector']

for metric in alpha_metrics:
    metric_path = f"{WORK_DIR}/07_diversity/core_metrics/{metric}.qza"
    output_path = f"{WORK_DIR}/07_diversity/{metric.replace('_vector','')}_kruskal.qzv"
    
    kruskal_cmd = f"""qiime diversity alpha-group-significance \
  --i-alpha-diversity {metric_path} \
  --m-metadata-file {METADATA_FINAL} \
  --o-visualization {output_path}"""
    
    import os
    if os.path.exists(metric_path):
        run_qiime(kruskal_cmd, f"Kruskal-Wallis test: {metric.replace('_vector','')}")
    else:
        print(f"⚠️  {metric_path} not found — skipping")

print()
print("📊 View alpha diversity significance tests at https://view.qiime2.org:")
for metric in alpha_metrics:
    print(f"   {WORK_DIR}/07_diversity/{metric.replace('_vector','')}_kruskal.qzv")


In [ ]:
# @title Step 6.4 — Plot alpha diversity (Python) { display-mode: "form" }
import subprocess, os, pandas as pd, matplotlib.pyplot as plt
from scipy import stats

alpha_metrics_files = {
    'Shannon Index':     f"{WORK_DIR}/07_diversity/core_metrics/shannon_vector.qza",
    'Observed Features': f"{WORK_DIR}/07_diversity/core_metrics/observed_features_vector.qza",
    "Faith's PD":        f"{WORK_DIR}/07_diversity/core_metrics/faith_pd_vector.qza",
}

meta = pd.read_csv(METADATA_FINAL, sep='\t', index_col='sample-id')
groups = meta[FORMULA.split('+')[0]]  # use first column for plotting

fig, axes = plt.subplots(1, len(alpha_metrics_files), figsize=(5 * len(alpha_metrics_files), 5))
if len(alpha_metrics_files) == 1:
    axes = [axes]

colors_list = ['#2e75b6', '#e8562a', '#4caf50', '#9c27b0', '#ff9800']
group_names = sorted(groups.unique())
color_map = {g: colors_list[i % len(colors_list)] for i, g in enumerate(group_names)}

for ax, (metric_name, metric_path) in zip(axes, alpha_metrics_files.items()):
    if not os.path.exists(metric_path):
        ax.text(0.5, 0.5, f'{metric_name}\nnot found', ha='center', va='center',
                transform=ax.transAxes)
        continue
    
    # Export metric
    export_dir = metric_path.replace('.qza', '_export')
    subprocess.run(f"source /opt/miniforge/etc/profile.d/conda.sh && conda activate {QIIME_ENV} && "
                   f"qiime tools export --input-path {metric_path} --output-path {export_dir}",
                   shell=True, executable='/bin/bash', capture_output=True)
    
    tsv_files = [f for f in os.listdir(export_dir) if f.endswith('.tsv')] if os.path.exists(export_dir) else []
    if not tsv_files:
        ax.text(0.5, 0.5, 'Export failed', ha='center', va='center', transform=ax.transAxes)
        continue
    
    div_df = pd.read_csv(f"{export_dir}/{tsv_files[0]}", sep='\t', index_col=0, skiprows=[1])
    div_col = div_df.columns[0]
    merged = div_df.merge(groups.rename('group'), left_index=True, right_index=True)
    
    # Box plot
    grp_col = 'group'
    grp_data = [merged[merged[grp_col] == g][div_col].dropna().values for g in group_names]
    bp = ax.boxplot(grp_data, patch_artist=True, widths=0.5,
                    medianprops={'color': 'white', 'linewidth': 2})
    for patch, gname in zip(bp['boxes'], group_names):
        patch.set_facecolor(color_map[gname])
        patch.set_alpha(0.85)
    
    # Strip plot (individual points)
    for i, (g, vals) in enumerate(zip(group_names, grp_data)):
        jitter = [i + 1 + (j - 0.5) * 0.25 for j in [0.3, 0.5, 0.7, 0.2, 0.8, 0.4, 0.6][:len(vals)]]
        ax.scatter(jitter[:len(vals)], vals, color=color_map[g], alpha=0.6, zorder=3, s=40)
    
    # Stats (Kruskal-Wallis if >2 groups, Mann-Whitney if 2)
    valid = [v for v in grp_data if len(v) > 0]
    if len(valid) == 2 and all(len(v) >= 3 for v in valid):
        stat, pval = stats.mannwhitneyu(*valid, alternative='two-sided')
        ax.set_title(f"{metric_name}\np = {pval:.3f}", fontweight='bold')
    elif len(valid) > 2:
        stat, pval = stats.kruskal(*valid)
        ax.set_title(f"{metric_name}\nKruskal-Wallis p = {pval:.3f}", fontweight='bold')
    else:
        ax.set_title(metric_name, fontweight='bold')
    
    ax.set_xticklabels(group_names, rotation=30, ha='right')
    ax.set_ylabel(metric_name)
    ax.set_xlabel('')

plt.suptitle('Alpha Diversity by Group', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(f"{WORK_DIR}/09_figures/alpha_diversity.pdf", bbox_inches='tight')
plt.show()
print(f"✅ Saved: {WORK_DIR}/09_figures/alpha_diversity.pdf")


In [ ]:
# @title Step 6.5 — Beta diversity (PCoA plots) { display-mode: "form" }
beta_metrics = {
    'bray_curtis': 'Bray-Curtis dissimilarity',
    'unweighted_unifrac': 'Unweighted UniFrac',
    'weighted_unifrac': 'Weighted UniFrac'
}

for metric, label in beta_metrics.items():
    pcoa_path = f"{WORK_DIR}/07_diversity/core_metrics/{metric}_pcoa_results.qza"
    output_path = f"{WORK_DIR}/07_diversity/{metric}_emperor.qzv"
    
    import os
    if os.path.exists(pcoa_path):
        emperor_cmd = f"""qiime emperor plot \
  --i-pcoa {pcoa_path} \
  --m-metadata-file {METADATA_FINAL} \
  --o-visualization {output_path}"""
        run_qiime(emperor_cmd, f"Emperor PCoA plot: {label}")
    else:
        print(f"⚠️  {metric} PCoA not found — run Step 6.2 first")

print()
print("📊 View interactive 3D PCoA plots at https://view.qiime2.org")
for metric in beta_metrics:
    print(f"   {WORK_DIR}/07_diversity/{metric}_emperor.qzv")


In [ ]:
# @title Step 6.6 — Plot PCoA in Python (publication-quality) { display-mode: "form" }
import subprocess, os, pandas as pd, numpy as np, matplotlib.pyplot as plt
from matplotlib.patches import Ellipse
import matplotlib.transforms as transforms

meta = pd.read_csv(METADATA_FINAL, sep='\t', index_col='sample-id')
group_col = FORMULA.split('+')[0]
group_names = sorted(meta[group_col].unique())
colors_list = ['#2e75b6', '#e8562a', '#4caf50', '#9c27b0', '#ff9800', '#795548']
color_map = {g: colors_list[i % len(colors_list)] for i, g in enumerate(group_names)}

metrics_to_plot = ['bray_curtis', 'unweighted_unifrac', 'weighted_unifrac']
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for ax, metric in zip(axes, metrics_to_plot):
    pcoa_path = f"{WORK_DIR}/07_diversity/core_metrics/{metric}_pcoa_results.qza"
    export_dir = pcoa_path.replace('.qza', '_export')
    
    if not os.path.exists(pcoa_path):
        ax.text(0.5, 0.5, f'{metric}\nnot available', ha='center', va='center',
                transform=ax.transAxes, fontsize=10)
        ax.set_title(metric.replace('_', ' ').title())
        continue
    
    subprocess.run(f"source /opt/miniforge/etc/profile.d/conda.sh && conda activate {QIIME_ENV} && "
                   f"qiime tools export --input-path {pcoa_path} --output-path {export_dir}",
                   shell=True, executable='/bin/bash', capture_output=True)
    
    ordination_file = f"{export_dir}/ordination.txt"
    if not os.path.exists(ordination_file):
        ax.text(0.5, 0.5, 'Export failed', ha='center', va='center', transform=ax.transAxes)
        continue
    
    # Parse ordination file
    coords = {}; proportion_explained = []
    with open(ordination_file) as f:
        lines = f.readlines()
    
    in_coords = False
    for line in lines:
        if line.startswith('Proportion explained'):
            try:
                vals = lines[lines.index(line)+2].strip().split('\t')
                proportion_explained = [float(v)*100 for v in vals]
            except:
                pass
        if line.startswith('Site'):
            in_coords = True; continue
        if in_coords and line.strip() == '':
            break
        if in_coords:
            parts = line.strip().split('\t')
            if len(parts) >= 3:
                try:
                    coords[parts[0]] = [float(parts[1]), float(parts[2])]
                except:
                    pass
    
    if not coords:
        ax.text(0.5, 0.5, 'Could not parse\nordination', ha='center', va='center',
                transform=ax.transAxes)
        continue
    
    coord_df = pd.DataFrame(coords, index=['PC1','PC2']).T
    merged = coord_df.merge(meta[[group_col]], left_index=True, right_index=True, how='inner')
    
    for g in group_names:
        sub = merged[merged[group_col] == g]
        ax.scatter(sub['PC1'], sub['PC2'], c=color_map[g], label=g, s=80, alpha=0.85,
                   edgecolors='white', linewidths=0.5, zorder=3)
        # 95% confidence ellipse
        if len(sub) >= 3:
            cov = np.cov(sub['PC1'], sub['PC2'])
            vals, vecs = np.linalg.eigh(cov)
            order = vals.argsort()[::-1]
            vals, vecs = vals[order], vecs[:, order]
            angle = np.degrees(np.arctan2(*vecs[:, 0][::-1]))
            width, height = 2 * 1.96 * np.sqrt(vals)
            ell = Ellipse(xy=(sub['PC1'].mean(), sub['PC2'].mean()),
                          width=width, height=height, angle=angle,
                          color=color_map[g], alpha=0.12)
            ax.add_patch(ell)
    
    xlab = f"PC1 ({proportion_explained[0]:.1f}%)" if proportion_explained else "PC1"
    ylab = f"PC2 ({proportion_explained[1]:.1f}%)" if len(proportion_explained) > 1 else "PC2"
    ax.set_xlabel(xlab); ax.set_ylabel(ylab)
    ax.set_title(metric.replace('_', ' ').title().replace('Unifrac', 'UniFrac'), fontweight='bold')
    ax.axhline(0, color='lightgrey', lw=0.5); ax.axvline(0, color='lightgrey', lw=0.5)
    ax.legend(frameon=False, fontsize=9)

plt.suptitle('Beta Diversity — PCoA', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(f"{WORK_DIR}/09_figures/pcoa_plots.pdf", bbox_inches='tight')
plt.show()
print(f"✅ Saved: {WORK_DIR}/09_figures/pcoa_plots.pdf")


In [ ]:
# @title Step 6.7 — PERMANOVA (beta diversity significance) { display-mode: "form" }
beta_matrices = {
    'bray_curtis': 'Bray-Curtis',
    'unweighted_unifrac': 'Unweighted UniFrac',
    'weighted_unifrac': 'Weighted UniFrac'
}

group_col = FORMULA.split('+')[0]

for metric, label in beta_matrices.items():
    dm_path = f"{WORK_DIR}/07_diversity/core_metrics/{metric}_distance_matrix.qza"
    output_path = f"{WORK_DIR}/07_diversity/{metric}_permanova.qzv"
    
    import os
    if os.path.exists(dm_path):
        perm_cmd = f"""qiime diversity beta-group-significance \
  --i-distance-matrix {dm_path} \
  --m-metadata-file {METADATA_FINAL} \
  --m-metadata-column {group_col} \
  --p-method permanova \
  --p-permutations 999 \
  --o-visualization {output_path}"""
        run_qiime(perm_cmd, f"PERMANOVA test: {label}")
    else:
        print(f"⚠️  Distance matrix not found for {metric}")

print()
print("📊 View PERMANOVA results at https://view.qiime2.org")


In [ ]:
# @title Step 6.8 — Rarefaction curves { display-mode: "form" }
rarefy_cmd = f"""qiime diversity alpha-rarefaction \
  --i-table {WORK_DIR}/05_taxonomy/feature_table_final.qza \
  --i-phylogeny {WORK_DIR}/06_phylogeny/rooted_tree.qza \
  --p-max-depth {SAMPLING_DEPTH} \
  --m-metadata-file {METADATA_FINAL} \
  --o-visualization {WORK_DIR}/07_diversity/rarefaction_curves.qzv"""

run_qiime(rarefy_cmd, "Generating rarefaction curves")

print()
print("📊 View rarefaction curves at https://view.qiime2.org:")
print(f"   {WORK_DIR}/07_diversity/rarefaction_curves.qzv")
print("   → Check that curves plateau — confirms adequate sequencing depth")


---
## Section 7 — Differential Abundance Analysis (ANCOM-BC) 📊

**What is ANCOM-BC?**  
ANCOM-BC (Analysis of Compositions of Microbiomes with Bias Correction) identifies taxa that are significantly more or less abundant between groups, while correcting for compositional biases.

**Example uses:**
- Which genera are enriched in fountain water during student activity vs. baseline?
- Which bacteria survive after disinfectant wipe treatment?


In [ ]:
# @title Step 7.1 — Collapse taxonomy to genus level { display-mode: "form" }
collapse_cmd = f"""qiime taxa collapse \
  --i-table {WORK_DIR}/05_taxonomy/feature_table_final.qza \
  --i-taxonomy {WORK_DIR}/05_taxonomy/taxonomy.qza \
  --p-level 6 \
  --o-collapsed-table {WORK_DIR}/08_ancombc/table_genus.qza"""

run_qiime(collapse_cmd, "Collapsing feature table to genus level (Level 6)")
print("   ✅ Level 6 = Genus in SILVA taxonomy")


In [ ]:
# @title Step 7.2 — Run ANCOM-BC { display-mode: "form" }
print(f"Formula: {FORMULA}")
print("⚠️  Formula must match column names in your metadata exactly")
print()

ancombc_cmd = f"""qiime composition ancombc \
  --i-table {WORK_DIR}/08_ancombc/table_genus.qza \
  --m-metadata-file {METADATA_FINAL} \
  --p-formula '{FORMULA}' \
  --o-differentials {WORK_DIR}/08_ancombc/ancombc_results.qza"""

run_qiime(ancombc_cmd, "Running ANCOM-BC differential abundance test")


In [ ]:
# @title Step 7.3 — Visualize ANCOM-BC results { display-mode: "form" }
da_viz_cmd = f"""qiime composition da-barplot \
  --i-data {WORK_DIR}/08_ancombc/ancombc_results.qza \
  --o-visualization {WORK_DIR}/08_ancombc/da_barplot.qzv"""

run_qiime(da_viz_cmd, "Generating differential abundance bar plot")

print()
print("📊 View ANCOM-BC results at https://view.qiime2.org:")
print(f"   {WORK_DIR}/08_ancombc/da_barplot.qzv")
print("   → Positive log-fold change = enriched in reference group")
print("   → Negative log-fold change = depleted in reference group")


In [ ]:
# @title Step 7.4 — Volcano plot of differential abundance { display-mode: "form" }
import subprocess, os, pandas as pd, numpy as np, matplotlib.pyplot as plt

# Export ANCOM-BC results
export_dir = f"{WORK_DIR}/08_ancombc/ancombc_export"
export_cmd = f"qiime tools export --input-path {WORK_DIR}/08_ancombc/ancombc_results.qza --output-path {export_dir}"
subprocess.run(f"source /opt/miniforge/etc/profile.d/conda.sh && conda activate {QIIME_ENV} && {export_cmd}",
               shell=True, executable='/bin/bash', capture_output=True)

# Look for exported files
if os.path.exists(export_dir):
    files = os.listdir(export_dir)
    print(f"Exported files: {files}")
    
    tsv_files = [f for f in files if f.endswith('.tsv')]
    if tsv_files:
        df = pd.read_csv(f"{export_dir}/{tsv_files[0]}", sep='\t')
        print(f"\nLoaded: {df.shape[0]} taxa")
        print(f"Columns: {list(df.columns)}")
        
        # Try to plot a volcano
        lfc_cols = [c for c in df.columns if 'lfc' in c.lower() or 'log' in c.lower()]
        pval_cols = [c for c in df.columns if 'q_val' in c.lower() or 'p_val' in c.lower() or 'pval' in c.lower()]
        
        if lfc_cols and pval_cols:
            lfc_col = lfc_cols[0]
            pval_col = pval_cols[0]
            
            df = df.dropna(subset=[lfc_col, pval_col])
            df['-log10p'] = -np.log10(df[pval_col].clip(lower=1e-300))
            
            sig = df[pval_col] < 0.05
            enriched = sig & (df[lfc_col] > 0)
            depleted = sig & (df[lfc_col] < 0)
            
            fig, ax = plt.subplots(figsize=(9, 6))
            ax.scatter(df.loc[~sig, lfc_col], df.loc[~sig, '-log10p'],
                       color='lightgrey', s=40, alpha=0.6, label='Not significant')
            ax.scatter(df.loc[enriched, lfc_col], df.loc[enriched, '-log10p'],
                       color='#e8562a', s=60, alpha=0.8, label=f'Enriched (n={enriched.sum()})')
            ax.scatter(df.loc[depleted, lfc_col], df.loc[depleted, '-log10p'],
                       color='#2e75b6', s=60, alpha=0.8, label=f'Depleted (n={depleted.sum()})')
            
            # Label top taxa
            top_taxa = df[sig].nlargest(8, '-log10p')
            for _, row in top_taxa.iterrows():
                name = str(row.get('id', '')).split(';')[-1].strip()[:20]
                ax.annotate(name, (row[lfc_col], row['-log10p']),
                           fontsize=7, ha='left', va='bottom',
                           xytext=(3, 3), textcoords='offset points')
            
            ax.axhline(-np.log10(0.05), color='red', linestyle='--', lw=1, alpha=0.5,
                      label='p = 0.05')
            ax.axvline(0, color='grey', lw=0.8)
            ax.set_xlabel(f'Log2 fold change ({lfc_col})')
            ax.set_ylabel('-log₁₀(q-value)')
            ax.set_title('ANCOM-BC Differential Abundance — Volcano Plot', fontweight='bold')
            ax.legend(frameon=False, fontsize=9)
            
            plt.tight_layout()
            plt.savefig(f"{WORK_DIR}/09_figures/ancombc_volcano.pdf", bbox_inches='tight')
            plt.show()
            print(f"✅ Saved: {WORK_DIR}/09_figures/ancombc_volcano.pdf")
        else:
            print(f"⚠️  Could not identify LFC/p-value columns. Available: {list(df.columns)}")
    else:
        print(f"⚠️  No TSV files found in export directory: {files}")
else:
    print("⚠️  Run Step 7.2 first")


---
## Section 8 — Export Results & Summary 📦

Package all your results and figures for download or poster preparation.


In [ ]:
# @title Step 8.1 — Export all QIIME 2 artifacts to TSV/BIOM { display-mode: "form" }
import subprocess, os

exports = [
    (f"{WORK_DIR}/05_taxonomy/feature_table_final.qza",     f"{WORK_DIR}/exports/feature_table",    "Feature table"),
    (f"{WORK_DIR}/05_taxonomy/taxonomy.qza",                f"{WORK_DIR}/exports/taxonomy",          "Taxonomy"),
    (f"{WORK_DIR}/04_dada2/rep_seqs.qza",                   f"{WORK_DIR}/exports/rep_seqs",          "Representative sequences"),
    (f"{WORK_DIR}/04_dada2/denoising_stats.qza",            f"{WORK_DIR}/exports/denoising_stats",   "Denoising stats"),
    (f"{WORK_DIR}/06_phylogeny/rooted_tree.qza",            f"{WORK_DIR}/exports/rooted_tree",       "Rooted tree"),
]

os.makedirs(f"{WORK_DIR}/exports", exist_ok=True)

for input_path, output_path, label in exports:
    if os.path.exists(input_path):
        result = subprocess.run(
            f"source /opt/miniforge/etc/profile.d/conda.sh && conda activate {QIIME_ENV} && "
            f"qiime tools export --input-path {input_path} --output-path {output_path}",
            shell=True, executable='/bin/bash', capture_output=True, text=True
        )
        status = "✅" if result.returncode == 0 else "❌"
        print(f"{status} {label}: {output_path}")
    else:
        print(f"⚠️  Skipping {label} — artifact not found")

# Convert feature table to TSV
biom_path = f"{WORK_DIR}/exports/feature_table/feature-table.biom"
tsv_path  = f"{WORK_DIR}/exports/feature_table/feature-table.tsv"
if os.path.exists(biom_path):
    subprocess.run(f"biom convert -i {biom_path} -o {tsv_path} --to-tsv", shell=True)
    print(f"✅ Feature table TSV: {tsv_path}")


In [ ]:
# @title Step 8.2 — Zip all figures and copy to Drive { display-mode: "form" }
import subprocess, os, shutil

# Copy all figures to Drive
drive_figures_dir = f"{BASE_DIR}/analysis_results/figures"
os.makedirs(drive_figures_dir, exist_ok=True)

for f in os.listdir(f"{WORK_DIR}/09_figures"):
    src = f"{WORK_DIR}/09_figures/{f}"
    dst = f"{drive_figures_dir}/{f}"
    shutil.copy2(src, dst)
    print(f"✅ Copied: {f}")

# Zip and copy QZV visualizations to Drive
drive_qzv_dir = f"{BASE_DIR}/analysis_results/qiime2_visualizations"
os.makedirs(drive_qzv_dir, exist_ok=True)

import glob
qzv_files = glob.glob(f"{WORK_DIR}/**/*.qzv", recursive=True)
for qzv in qzv_files:
    dst = f"{drive_qzv_dir}/{os.path.basename(qzv)}"
    shutil.copy2(qzv, dst)

print(f"\n✅ All results saved to Google Drive: {BASE_DIR}/analysis_results/")
print(f"   Figures: {len(os.listdir(drive_figures_dir))} files")
print(f"   QZV visualizations: {len(qzv_files)} files")


In [ ]:
# @title Step 8.3 — Print analysis summary { display-mode: "form" }
import os, glob

print("=" * 60)
print(f"  ANALYSIS COMPLETE — {PROJECT_NAME}")
print("=" * 60)

sections = [
    ("Import",        f"{WORK_DIR}/02_import/demux.qza"),
    ("Cutadapt",      f"{WORK_DIR}/03_cutadapt/demux_trimmed.qza"),
    ("DADA2 table",   f"{WORK_DIR}/04_dada2/feature_table.qza"),
    ("DADA2 seqs",    f"{WORK_DIR}/04_dada2/rep_seqs.qza"),
    ("Taxonomy",      f"{WORK_DIR}/05_taxonomy/taxonomy.qza"),
    ("Filtered table",f"{WORK_DIR}/05_taxonomy/feature_table_final.qza"),
    ("Tree",          f"{WORK_DIR}/06_phylogeny/rooted_tree.qza"),
    ("Diversity",     f"{WORK_DIR}/07_diversity/core_metrics"),
    ("ANCOM-BC",      f"{WORK_DIR}/08_ancombc/ancombc_results.qza"),
]

print("\nPipeline artifacts:")
for label, path in sections:
    status = "✅" if os.path.exists(path) else "⬜ (not run)"
    print(f"  {status} {label:20s} {path.replace(WORK_DIR, '...')}")

qzv_files = glob.glob(f"{WORK_DIR}/**/*.qzv", recursive=True)
pdf_files  = glob.glob(f"{WORK_DIR}/09_figures/*.pdf")
print(f"\n📊 Visualizations:  {len(qzv_files)} QZV files")
print(f"📈 Figures:          {len(pdf_files)} PDF files")
print()
print("📌 Next steps:")
print("   1. Download QZV files and view at https://view.qiime2.org")
print("   2. Download PDF figures for your poster")
print("   3. Set SAMPLING_DEPTH based on feature table summary (Step 4.2)")
print("   4. Adjust TRUNC_LEN_F/R based on quality plots (Step 3.2)")
print("   5. Re-run diversity steps (Section 6) with updated depth")


---
## Section 9 — Instructor Notes & Troubleshooting 📚

### Common issues and fixes

| Problem | Likely cause | Fix |
|---------|-------------|-----|
| DADA2 fails or returns very few ASVs | Truncation lengths too aggressive | Lower `TRUNC_LEN_F`/`TRUNC_LEN_R` — ensure overlap of at least 20 bp for paired-end |
| Taxonomy step takes forever | Colab RAM exceeded | Reduce batch size or use pre-trained V4-specific classifier |
| `conda: command not found` | Session restarted | Re-run Section 1 from the top |
| Diversity step excludes all samples | `SAMPLING_DEPTH` too high | Lower it; check minimum sample depth in feature table summary (Step 4.2) |
| ANCOM-BC formula error | Column name mismatch | Check `metadata.tsv` column names — use underscores, no spaces |
| Empty PCoA plots | Core metrics failed | Verify sampling depth > 0 and feature table is not empty after filtering |

---

### Choosing truncation lengths (Step 3.2 → Section 2)

Open `demux_summary.qzv` at [view.qiime2.org](https://view.qiime2.org) and go to the **Interactive Quality Plot** tab:
- Find where the **median quality drops below Q25**
- Set `TRUNC_LEN_F` to that position for forward reads
- Set `TRUNC_LEN_R` to that position for reverse reads
- **Critical:** `(TRUNC_LEN_F + TRUNC_LEN_R) - amplicon_length ≥ 20` for proper merging
- For V4 region (515F/806R, ~253 bp amplicon): `TRUNC_LEN_F=250, TRUNC_LEN_R=200` is often reasonable

---

### Choosing sampling depth (Step 4.2 → Section 2)

From `feature_table_summary.qzv`, look at **Interactive Sample Detail**:
- Sort by frequency (ascending)
- Choose a depth that retains ≥ 80% of your samples
- **Exclude** outlier samples with very low reads (add to `EXCLUDE_IDS` in Section 2)
- A good rule of thumb: median depth × 0.3 as minimum

---

### Interpreting diversity metrics

**Alpha diversity (within-sample):**
- Higher Shannon / Observed features = more diverse community
- Compare groups statistically with Kruskal-Wallis (Step 6.3)

**Beta diversity (between-sample):**
- Points closer together on PCoA = more similar communities
- PERMANOVA p < 0.05 means groups are significantly different in composition
- Check both Bray-Curtis (abundance-based) and UniFrac (phylogenetically informed)

---

### Primer sequences used in this curriculum

| Primer | Sequence |
|--------|----------|
| 515F (Parada) | `GTGYCAGCMGCCGCGGTAA` |
| 806R (Apprill) | `GGACTACNVGGGTWTCTAAT` |

---

### Runtime notes for students
- Section 1 (install): ~20 min — do this before your lab session
- Section 4 (DADA2): ~30–90 min — submit and come back
- Sections 5–7: ~10–20 min each
- Save to Drive frequently; Colab disconnects after ~90 min idle
